# Generate `cleaned_cell_labels_meta_tea_seq.csv`

Generates the cell type metadata CSV directly from `rna.h5ad` obs, ensuring the barcodes and cell types are always in sync with the AnnData files.

**Run this whenever `rna.h5ad` is regenerated.**

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
from pathlib import Path

In [ ]:
data_dir = Path("../data")

rna = sc.read_h5ad(data_dir / "rna.h5ad")
print("Loaded rna.h5ad:", rna.shape)
print("obs columns:", list(rna.obs.columns))

In [ ]:
meta = pd.DataFrame(
    {"CellType": rna.obs["celltype"].values},
    index=rna.obs_names
)

out_path = data_dir / "cleaned_cell_labels_meta_tea_seq.csv"
meta.to_csv(out_path)
print(f"Written: {out_path}  ({len(meta)} cells)")
print(meta["CellType"].value_counts())

In [ ]:
# Verify: reload and confirm zero mismatches
meta_check = pd.read_csv(out_path, index_col=0)
cell_ids = rna.obs_names.to_numpy()
meta_bc  = meta_check.index.to_numpy()

set_mismatch   = len(set(cell_ids) - set(meta_bc))
order_mismatch = int(np.sum(cell_ids != meta_bc))

assert set_mismatch == 0,   f"Set mismatch: {set_mismatch} barcodes differ"
assert order_mismatch == 0, f"Order mismatch: {order_mismatch} rows in different positions"

print("Verification passed: 0 set mismatches, 0 row-order mismatches")